
# SOHBusBox – Casablanca (Sidi Maârouf) – QA & Diagnostics

This notebook adds **testing utilities** to verify that your scenario is set up correctly:
- load and preview **BusDriver** and **Traveler** trips,
- classify whether a traveler **used the bus** (spatial overlap),
- check **speeds** and **lengths**,
- validate trips **stay inside** the AOI,
- compute simple **headways** and **bus speeds**,
- map traveler **origins/targets → nearest stops** (boarding potential).

> Edit the *paths* in the next cell if your file names differ.


In [ ]:

from pathlib import Path
import json
import math
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union, linemerge
from shapely.geometry import LineString, MultiLineString, Point
import matplotlib.pyplot as plt

# ---- Paths (adjust to your project) ----
TRAVELER_TRIPS = Path("CasablancaCityBox/PassengerTraveler_trips.geojson")  # or: output/all_agents_debug_trips.geojson
BUS_TRIPS      = Path("CasablancaCityBox/BusDriver_trips.geojson")
BUS_ROUTE      = Path("CasablancaCityBox/resources/routes.geojson")                 # fallback corridor if BUS_TRIPS is missing
WALK_GRAPH     = Path("CasablancaCityBox/resources/casa_sidi_maarouf_walk_graph.geojson")
DRIVE_GRAPH    = Path("CasablancaCityBox/resources/casa_sidi_maarouf_drive_graph.geojson")
AOI_FILE       = Path("CasablancaCityBox/resources/casa_sidi_maarouf_aoi.geojson")
STATIONS_FILE  = Path("CasablancaCityBox/resources/stations.geojson")  # optional

# Helper: safe loader
def load_gdf(p):
    if not p or not Path(p).exists():
        return None
    g = gpd.read_file(p)
    if g.crs is None or g.crs.to_epsg() != 4326:
        g = g.to_crs(4326)
    return g

trav = load_gdf(TRAVELER_TRIPS)
bus  = load_gdf(BUS_TRIPS)
route= load_gdf(BUS_ROUTE)
walk = load_gdf(WALK_GRAPH)
drive= load_gdf(DRIVE_GRAPH)
aoi  = load_gdf(AOI_FILE)
stops= load_gdf(STATIONS_FILE)

print("Loaded:",
      "\n  travelers:", trav.shape if trav is not None else None,
      "\n  bus trips:", bus.shape if bus is not None else None,
      "\n  route:", route.shape if route is not None else None,
      "\n  walk:", walk.shape if walk is not None else None,
      "\n  drive:", drive.shape if drive is not None else None,
      "\n  aoi:", aoi.shape if aoi is not None else None,
      "\n  stations:", stops.shape if stops is not None else None)


In [ ]:

# Static extent preview: AOI + route + first bus + a sample of travelers
fig = plt.figure(figsize=(8,8))
if aoi is not None: aoi.plot(ax=plt.gca(), alpha=0.1)
if route is not None: route.plot(ax=plt.gca(), linewidth=1)
if bus is not None and len(bus)>0: bus.iloc[:5].plot(ax=plt.gca(), linewidth=1)
if trav is not None and len(trav)>0: trav.sample(min(10, len(trav)), random_state=1).plot(ax=plt.gca(), linewidth=0.7)
plt.title("AOI / Route / Bus / Travelers (sample)")
plt.xlabel("lon"); plt.ylabel("lat")
plt.show()


In [ ]:

# Build a bus corridor buffer (meters) from bus trips if available, else from route
def choose_metric_crs(*gdfs):
    for g in gdfs:
        if g is not None and not g.empty:
            try:
                return g.estimate_utm_crs()
            except Exception:
                pass
    return "EPSG:3857"

def build_corridor(gdf_lines, buffer_m=25.0):
    if gdf_lines is None or gdf_lines.empty:
        return None, None
    metric = gdf_lines.to_crs(choose_metric_crs(gdf_lines))
    lines = [geom for geom in metric.geometry if isinstance(geom, (LineString, MultiLineString))]
    if not lines:
        return None, metric.crs
    union = unary_union(lines)
    corr = gpd.GeoSeries([union], crs=metric.crs).buffer(buffer_m).iloc[0]
    return corr, metric.crs

bus_corr, metric_crs = (build_corridor(bus, 25.0) if bus is not None and not bus.empty else build_corridor(route, 25.0))
print("metric_crs:", metric_crs)


In [ ]:

def frac_len(a, b):
    if a is None or b is None or a.is_empty:
        return 0.0
    inter = a.intersection(b)
    try:
        return inter.length / a.length if a.length>0 else 0.0
    except Exception:
        return 0.0

def parse_duration_seconds(v):
    import pandas as pd
    if v is None: return None
    if isinstance(v, (int,float)): return float(v)
    s = str(v)
    if ":" in s:
        try:
            h,m,sec = s.split(":")
            return int(h)*3600 + int(m)*60 + float(sec)
        except Exception:
            return None
    try:
        return float(s)
    except Exception:
        return None

report = []
trav_used_idx = []
trav_non_idx = []
if trav is not None and bus_corr is not None:
    trav_m = trav.to_crs(metric_crs)
    buf = bus_corr
    for i, r in trav_m.iterrows():
        g = r.geometry
        if g is None or g.is_empty: 
            trav_non_idx.append(i)
            report.append(dict(idx=i, reason="empty", bus_overlap=0.0, length_m=0.0))
            continue
        L = g.length
        bus_frac = frac_len(g, buf)
        dur = parse_duration_seconds(r.get("Duration", None))
        dist = r.get("DistanceTraveled", None)
        try: dist = float(dist)
        except Exception: dist = L
        speed = (dist/dur) if (dur and dur>0) else None
        ok = (L >= 500) and (bus_frac >= 0.45)
        report.append(dict(idx=i, length_m=float(L), bus_frac=float(bus_frac), speed_m_s=None if speed is None else float(speed), ok=ok))
        (trav_used_idx if ok else trav_non_idx).append(i)

    import pandas as pd
    rep = pd.DataFrame(report)
    display(rep.head(12))
else:
    print("Not enough data to classify (need traveler trips and bus/route corridor).")


In [ ]:

from pathlib import Path
outdir = Path("verify_bus"); outdir.mkdir(parents=True, exist_ok=True)
if trav is not None and len(trav_used_idx)>0:
    trav.iloc[trav_used_idx].to_file(outdir/"traveler_bus_used.geojson", driver="GeoJSON")
if trav is not None and len(trav_non_idx)>0:
    trav.iloc[trav_non_idx].to_file(outdir/"traveler_non_bus.geojson", driver="GeoJSON")
print("Wrote:", outdir)


In [ ]:

# Simple bus summaries (works if BusDriver trips contain StableId + Duration/DistanceTraveled)
import numpy as np
if bus is not None and not bus.empty:
    df = bus.copy()
    # Normalize time if present
    def dur_s(v):
        try:
            return float(v) if isinstance(v,(int,float)) else (int(v.split(":")[0])*3600 + int(v.split(":")[1])*60 + float(v.split(":")[2]))
        except Exception:
            return None
    df["duration_s"] = df.get("Duration", None).apply(dur_s) if "Duration" in df else None
    df["distance_m"] = pd.to_numeric(df.get("DistanceTraveled", None), errors="coerce") if "DistanceTraveled" in df else None
    df["speed_m_s"]  = df.apply(lambda r: (r["distance_m"]/r["duration_s"]) if (r.get("duration_s") and r.get("distance_m")) else None, axis=1)

    # Plot speeds histogram
    s = df["speed_m_s"].dropna()
    if len(s)>0:
        plt.figure(figsize=(6,4))
        plt.hist(s, bins=20)
        plt.title("Bus average speeds (m/s)")
        plt.xlabel("m/s"); plt.ylabel("count")
        plt.show()
else:
    print("No bus trips to summarize.")


In [ ]:

# If your travelers spawn from POINT geometries near stops, check nearest stop distance
if stops is not None and trav is not None:
    # get first vertex of each traveler line as origin
    origins = trav.copy()
    def first_point(geom):
        try:
            if isinstance(geom, LineString):
                x,y = geom.coords[0]
                return Point(x,y)
            elif isinstance(geom, MultiLineString):
                ls = list(geom.geoms)[0]
                x,y = ls.coords[0]
                return Point(x,y)
        except Exception:
            return None
    origins["origin_pt"] = origins.geometry.apply(first_point)
    origins = origins.dropna(subset=["origin_pt"]).set_geometry("origin_pt")

    # project to metric
    o_m = origins.to_crs(metric_crs)
    s_m = stops.to_crs(metric_crs)

    # simple nearest distance
    from scipy.spatial import cKDTree
    import numpy as np
    stop_xy = np.vstack([s_m.geometry.x.values, s_m.geometry.y.values]).T
    tree = cKDTree(stop_xy)
    ori_xy = np.vstack([o_m.geometry.x.values, o_m.geometry.y.values]).T
    dist, idx = tree.query(ori_xy, k=1)
    plt.figure(figsize=(6,4))
    plt.hist(dist, bins=20)
    plt.title("Origin → nearest stop distance (m)")
    plt.xlabel("meters"); plt.ylabel("count")
    plt.show()
else:
    print("Provide STATIONS_FILE and traveler trips to run the stop proximity check.")
